# U23 — Cloud Storage & Databases (finish): Lab

### Real-world brief: making a plant-telemetry database fast & scalable

An industrial plant streams **200,000 sensor readings** from 30 machines into a database, and the analytics queries are getting slow. In this Part-2 lab you'll apply the scaling and performance techniques from the deck **hands-on** with SQLite + Python: **indexing & query plans**, a **cache-aside** layer, **read replicas** with lag, **sharding** across nodes, a **materialized view**, and basic **security** (PII masking, read-only access).

**Resources provided:** `plant_telemetry.csv` (≈200k readings) and `machines.csv` (asset master). Everything runs locally — but each technique maps directly to its cloud equivalent.

_Phase F — Data Engineering._

#objectives

Speed up queries with indexes and read the query plan

Build a cache-aside layer and measure the hit-rate speedup

Simulate read replicas and observe replication lag

Shard data across nodes and run a scatter-gather query

Add a materialized view and basic security (masking, read-only)

In [1]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def build_plant(tele_path="plant_telemetry.csv", mach_path="machines.csv", seed=232, verbose=False):
    """A larger industrial-IoT telemetry table + machine master — sized so that indexing,
    caching, sharding and materialized-view effects are actually MEASURABLE (U23 Part 2).

      machines.csv          30 machines (asset master)
      plant_telemetry.csv   ~200k sensor readings (the high-volume table)
    """
    rng = np.random.default_rng(seed)

    N_MACH = 30
    machines = pd.DataFrame({
        "machine_id": [f"M{i:03d}" for i in range(1, N_MACH + 1)],
        "line": rng.choice(["LINE_1", "LINE_2", "LINE_3"], N_MACH),
        "machine_type": rng.choice(["pump", "compressor", "conveyor", "press"], N_MACH),
        "install_year": rng.integers(2014, 2023, N_MACH),
        "criticality": rng.choice(["low", "medium", "high"], N_MACH, p=[0.4, 0.4, 0.2]),
    })
    machines.to_csv(mach_path, index=False)

    N = 200_000
    mids = machines.machine_id.values
    metric = rng.choice(["temp", "vibration", "pressure", "current"], N, p=[0.3, 0.3, 0.2, 0.2])
    base = {"temp": 68, "vibration": 2.6, "pressure": 6.2, "current": 31}
    sd = {"temp": 7, "vibration": 0.9, "pressure": 0.7, "current": 4}
    value = np.array([rng.normal(base[m], sd[m]) for m in metric]).round(3)
    ts = pd.Timestamp("2024-06-01") + pd.to_timedelta(np.sort(rng.uniform(0, 14 * 86400, N)), unit="s")
    tele = pd.DataFrame({
        "reading_id": np.arange(N),
        "ts": ts.round("s"),
        "machine_id": rng.choice(mids, N),
        "metric": metric,
        "value": value,
        "quality_flag": rng.choice(["ok", "suspect"], N, p=[0.97, 0.03]),
    })
    tele.to_csv(tele_path, index=False)

    if verbose:
        print("machines:", machines.shape, "| telemetry:", tele.shape)
        print("distinct machines:", tele.machine_id.nunique(), "| metrics:", tele.metric.nunique())
        print("date span:", tele.ts.min(), "->", tele.ts.max())
    return machines, tele

if not (os.path.exists('plant_telemetry.csv') and os.path.exists('machines.csv')):
    build_plant(); print('Generated source files.')
else:
    print('Found the provided source files.')

Generated source files.


In [2]:
import pandas as pd, numpy as np, sqlite3, time, hashlib
tele = pd.read_csv('plant_telemetry.csv', parse_dates=['ts'])
mach = pd.read_csv('machines.csv')
print('telemetry:', tele.shape, '| machines:', mach.shape)
# load into a single SQLite 'database'
DB = 'plant.db'
if os.path.exists(DB): os.remove(DB)
con = sqlite3.connect(DB)
tele.assign(ts=tele.ts.astype(str)).to_sql('telemetry', con, index=False)
mach.to_sql('machines', con, index=False)
con.commit()
print('loaded into SQLite.')

def timed(sql, params=(), repeats=5):
    best = min(_t(sql, params) for _ in range(repeats)); return best
def _t(sql, params):
    t = time.perf_counter(); con.execute(sql, params).fetchall(); return (time.perf_counter()-t)*1000

telemetry: (200000, 6) | machines: (30, 5)
loaded into SQLite.


#1. Indexing & query plans

In [3]:
# -----------------------------------------------------------
# 🔹 1A. A FILTER QUERY: scan vs index
# -----------------------------------------------------------
q = "SELECT * FROM telemetry WHERE machine_id = ? AND metric = ?"
print('plan BEFORE:', con.execute('EXPLAIN QUERY PLAN ' + q, ('M007', 'temp')).fetchall()[0][-1])
t0 = timed(q, ('M007', 'temp'))
con.execute('CREATE INDEX idx_mach_metric ON telemetry(machine_id, metric)'); con.commit()
print('plan AFTER :', con.execute('EXPLAIN QUERY PLAN ' + q, ('M007', 'temp')).fetchall()[0][-1])
t1 = timed(q, ('M007', 'temp'))
print(f'query: {t0:.2f} ms (scan) -> {t1:.2f} ms (indexed) = {t0/max(t1,1e-6):.1f}x faster')

plan BEFORE: SCAN telemetry
plan AFTER : SEARCH telemetry USING INDEX idx_mach_metric (machine_id=? AND metric=?)
query: 22.53 ms (scan) -> 6.26 ms (indexed) = 3.6x faster


#### 🧪 EXERCISE 1 — A covering index
A *covering* index contains every column a query needs, so the database never touches the table.
1. Write a query that selects only `value` filtered by `machine_id` and `metric`. Time it with the existing index.
2. Create an index on `(machine_id, metric, value)` and re-time — the plan should say it uses the index without a table lookup.
3. In a comment, explain the trade-off (indexes speed reads but cost write time and storage).

In [4]:
# 1-2. covering index demo + timing
q_value = "SELECT value FROM telemetry WHERE machine_id = ? AND metric = ?"

# Time with existing index (idx_mach_metric)
print('plan BEFORE covering index:')
print(con.execute('EXPLAIN QUERY PLAN ' + q_value, ('M007', 'temp')).fetchall()[0][-1])
t_before_covering = timed(q_value, ('M007', 'temp'))

# Create covering index
con.execute('CREATE INDEX idx_mach_metric_value ON telemetry(machine_id, metric, value)')
con.commit()

# Re-time with covering index
print('\nplan AFTER covering index:')
print(con.execute('EXPLAIN QUERY PLAN ' + q_value, ('M007', 'temp')).fetchall()[0][-1])
t_after_covering = timed(q_value, ('M007', 'temp'))

print(f'query (value only): {t_before_covering:.2f} ms (partial index) -> {t_after_covering:.2f} ms (covering index) = {t_before_covering/max(t_after_covering,1e-6):.1f}x faster')


plan BEFORE covering index:
SEARCH telemetry USING INDEX idx_mach_metric (machine_id=? AND metric=?)

plan AFTER covering index:
SEARCH telemetry USING COVERING INDEX idx_mach_metric_value (machine_id=? AND metric=?)
query (value only): 8.38 ms (partial index) -> 1.26 ms (covering index) = 6.7x faster


# 3. index trade-off:
 Indexes speed up read queries by allowing the database to quickly locate data without scanning the entire table.
 However, they come with trade-offs:
 - Write performance: Every time data is inserted, updated, or deleted in the indexed columns, the index must also be updated, which adds overhead and slows down write operations.
 - Storage overhead: Indexes require additional disk space to store their structure.
 - Complexity: Managing indexes adds complexity to database administration.

#2. Cache-aside caching

In [5]:
# -----------------------------------------------------------
# 🔹 2A. A CACHE-ASIDE LAYER in front of an expensive query
# -----------------------------------------------------------
def expensive_query(machine_id):
    # an aggregation the app asks for repeatedly
    sql = 'SELECT metric, AVG(value) FROM telemetry WHERE machine_id=? GROUP BY metric'
    return tuple(con.execute(sql, (machine_id,)).fetchall())

cache = {}; hits = misses = 0
def get_machine_avgs(machine_id):
    global hits, misses
    if machine_id in cache:           # 1) check cache
        hits += 1; return cache[machine_id]
    misses += 1                        # 2) miss -> hit the DB
    result = expensive_query(machine_id)
    cache[machine_id] = result         # 3) fill the cache
    return result

import random
queries = [f'M{random.randint(1,30):03d}' for _ in range(2000)]   # repeated access pattern
t = time.perf_counter()
for mid in queries: get_machine_avgs(mid)
elapsed = (time.perf_counter()-t)*1000
print(f'2000 lookups in {elapsed:.0f} ms | hits={hits} misses={misses} | hit-rate={hits/2000:.1%}')
print('Only the first lookup per machine touches the DB; the rest are served from memory.')

2000 lookups in 35 ms | hits=1970 misses=30 | hit-rate=98.5%
Only the first lookup per machine touches the DB; the rest are served from memory.


#### 🧪 EXERCISE 2 — Invalidation & speedup
1. Measure the average time of a **cache hit** vs a **cache miss** (a miss runs `expensive_query`). Report the speedup.
2. Cache **invalidation**: write `record_reading(machine_id, ...)` that inserts a row AND evicts that machine's cache entry (so the next read recomputes). Show the entry is gone after a write.
3. In a comment, explain why stale caches are the classic caching bug, and what a TTL would add.

In [6]:
# 1-2. hit vs miss timing; write-with-invalidation

# Reset cache for accurate timing
cache = {}; hits = misses = 0

# Time a cache miss
expensive_machine_id = 'M001'
t_miss_start = time.perf_counter()
get_machine_avgs(expensive_machine_id)
t_miss = (time.perf_counter() - t_miss_start) * 1000
print(f'Cache MISS for {expensive_machine_id}: {t_miss:.2f} ms')

# Time a cache hit (same machine_id)
t_hit_start = time.perf_counter()
get_machine_avgs(expensive_machine_id)
t_hit = (time.perf_counter() - t_hit_start) * 1000
print(f'Cache HIT for {expensive_machine_id}: {t_hit:.2f} ms')

# Calculate speedup
if t_hit > 0: # Avoid division by zero
    speedup = t_miss / t_hit
    print(f'Cache hit is {speedup:.1f}x faster than a miss.')
else:
    print('Cache hit was extremely fast (near 0 ms), cannot calculate meaningful speedup.')

# --------------------------------------------------------------------------------

# 2. Cache invalidation: record_reading function
# Reset cache again for this part
cache = {}; hits = misses = 0

def record_reading(machine_id, ts, metric, value, quality_flag='ok'):
    # Invalidate the cache entry for this machine_id
    if machine_id in cache:
        del cache[machine_id]
        print(f'Invalidated cache for {machine_id} due to new reading.')

    # Insert the new reading into the database
    # Note: SQLite's auto-incrementing primary key handles reading_id if not provided
    con.execute(
        'INSERT INTO telemetry (ts, machine_id, metric, value, quality_flag) VALUES (?, ?, ?, ?, ?)',
        (str(ts), machine_id, metric, value, quality_flag)
    )
    con.commit()
    print(f'Recorded new reading for {machine_id}.')

# Demonstrate invalidation
m_to_invalidate = 'M002'
# Populate cache for M002
get_machine_avgs(m_to_invalidate)
print(f'M002 in cache BEFORE write: {m_to_invalidate in cache}')

# Simulate a new reading for M002
record_reading(m_to_invalidate, pd.Timestamp('2024-06-15 12:00:00'), 'pressure', 5.5, 'ok')

# Check if M002 is still in cache after the write
print(f'M002 in cache AFTER write and invalidation: {m_to_invalidate in cache}')


Cache MISS for M001: 2.25 ms
Cache HIT for M001: 0.12 ms
Cache hit is 18.8x faster than a miss.
M002 in cache BEFORE write: True
Invalidated cache for M002 due to new reading.
Recorded new reading for M002.
M002 in cache AFTER write and invalidation: False


# 3. cache staleness & TTL:
Cache staleness is a classic bug because applications might serve outdated data if the underlying source (database) changes but the cache is not updated or invalidated. This can lead to inconsistent views of data, incorrect decisions based on old information, and a poor user experience.
A Time-To-Live (TTL) mechanism would add automatic expiry to cached items. After a specified duration, a cached entry would be considered stale and automatically removed, forcing the next request for that data to go to the source. This helps mitigate the risk of serving excessively old data, providing a balance between performance and data freshness, especially for data that changes less frequently or where some degree of staleness is acceptable.

#3. Read replicas & replication lag

In [7]:
# -----------------------------------------------------------
# 🔹 3A. PRIMARY takes writes; a REPLICA serves reads (with lag)
# -----------------------------------------------------------
primary = sqlite3.connect('primary.db'); primary.execute('DROP TABLE IF EXISTS kv')
primary.execute('CREATE TABLE kv (machine_id TEXT PRIMARY KEY, status TEXT)')
primary.execute("INSERT INTO kv VALUES ('M007','ok')"); primary.commit()

def make_replica(src='primary.db', dst='replica.db'):
    import shutil; shutil.copy(src, dst); return sqlite3.connect(dst)
replica = make_replica()   # snapshot copy = a (lagging) read replica

# a WRITE goes to the primary only
primary.execute("UPDATE kv SET status='fault' WHERE machine_id='M007'"); primary.commit()
p = primary.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
r = replica.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f'read from PRIMARY: {p}  |  read from REPLICA: {r}  <- stale (replication lag)')
replica = make_replica()   # replication catches up (re-sync)
r2 = replica.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f'after replica re-syncs: {r2}  (eventual consistency)')

read from PRIMARY: fault  |  read from REPLICA: ok  <- stale (replication lag)
after replica re-syncs: fault  (eventual consistency)


#### 🧪 EXERCISE 3 — Read/write routing
1. Write a tiny router: `def query(sql, write=False)` that sends writes to `primary` and reads to `replica`. Use it to do one write then one read.
2. In a comment, explain when routing reads to replicas is safe (analytics, dashboards) and when it is dangerous (read-after-write where the user expects to see their own change immediately).

In [8]:
# 1. read/write router
def query(sql, params=(), write=False):
    if write:
        conn = primary
        print(f"Executing write on primary: {sql}")
    else:
        conn = replica
        print(f"Executing read on replica: {sql}")
    cursor = conn.execute(sql, params)
    conn.commit() # Commit for both reads and writes, although for reads it's a no-op
    return cursor.fetchall()

# Demonstrate usage:
# First, let's ensure the replica is up-to-date for a fresh start in the demo
replica = make_replica() # re-sync replica

# Perform a write operation using the router
print("\n--- Demonstrating write operation ---")
query("UPDATE kv SET status='active' WHERE machine_id='M007'", write=True)

# Read from primary to confirm write
primary_status = primary.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f"Primary status after write: {primary_status}")

# Read from replica immediately after write (should show replication lag)
replica_status_before_resync = replica.execute("SELECT status FROM kv WHERE machine_id='M007'").fetchone()[0]
print(f"Replica status immediately after write (lag): {replica_status_before_resync}")

# Re-sync replica and read again to show eventual consistency
replica = make_replica() # Re-sync replica
replica_status_after_resync = query("SELECT status FROM kv WHERE machine_id='M007'")
print(f"Replica status after re-sync (eventual consistency): {replica_status_after_resync[0][0]}")




--- Demonstrating write operation ---
Executing write on primary: UPDATE kv SET status='active' WHERE machine_id='M007'
Primary status after write: active
Replica status immediately after write (lag): fault
Executing read on replica: SELECT status FROM kv WHERE machine_id='M007'
Replica status after re-sync (eventual consistency): active


# 2. when replica reads are (un)safe:
- Routing reads to replicas is safe for analytical queries, dashboards, or reports where immediate consistency is not critical.
- For example, displaying historical trends or aggregated data can tolerate some replication lag.

- It is dangerous when the application requires strong read-after-write consistency, meaning a user expects to see their own changes immediately after they have made them.
- Examples include user profile updates, financial transactions, or any operation where a user's perceived state must be current.
- In these cases, reading from a potentially lagging replica can lead to a confusing or incorrect user experience, as they might not see their recently committed data.

#4. Sharding across nodes

In [9]:
# -----------------------------------------------------------
# 🔹 4A. SHARD the telemetry across N nodes by machine_id
# -----------------------------------------------------------
import zlib
N_SHARDS = 4
def shard_of(machine_id): return zlib.crc32(machine_id.encode()) % N_SHARDS

shards = [sqlite3.connect(f'shard_{i}.db') for i in range(N_SHARDS)]
for sc in shards:
    sc.execute('DROP TABLE IF EXISTS telemetry')
    sc.execute('CREATE TABLE telemetry (machine_id TEXT, metric TEXT, value REAL)')
rows = con.execute('SELECT machine_id, metric, value FROM telemetry').fetchall()
for mid, metric, val in rows:
    shards[shard_of(mid)].execute('INSERT INTO telemetry VALUES (?,?,?)', (mid, metric, val))
for sc in shards: sc.commit()
sizes = [sc.execute('SELECT COUNT(*) FROM telemetry').fetchone()[0] for sc in shards]
print('rows per shard:', sizes, '-> total', sum(sizes))

# a single-machine query hits ONE shard (routed); a global query is scatter-gather
one = shards[shard_of('M007')].execute("SELECT COUNT(*) FROM telemetry WHERE machine_id='M007'").fetchone()[0]
print('M007 lives on shard', shard_of('M007'), 'with', one, 'rows (routed read, no fan-out)')

rows per shard: [53096, 53359, 46973, 46573] -> total 200001
M007 lives on shard 0 with 6687 rows (routed read, no fan-out)


#### 🧪 EXERCISE 4 — Scatter-gather & skew
1. Compute the **global** average `value` per metric by querying **every** shard and combining the partial results (scatter-gather). Compare to the single-DB answer.
2. Show the downside of a bad shard key: count rows if you sharded by **metric** (only 4 distinct values) instead of machine_id — note how few shards would be used and why that causes hotspots.
3. In a comment, state the rule for picking a shard key (high cardinality, even access).

In [10]:
# 1-2. scatter-gather global average; metric-skew demonstration

# 1. Compute global average value per metric using scatter-gather
print("--- Global Average (Scatter-Gather) ---")
global_avg_data = {}

for i, sc in enumerate(shards):
    # Query each shard for partial aggregates (sum of value, count of values per metric)
    shard_results = sc.execute(
        'SELECT metric, SUM(value), COUNT(*) FROM telemetry GROUP BY metric'
    ).fetchall()
    for metric, sum_val, count_val in shard_results:
        if metric not in global_avg_data:
            global_avg_data[metric] = {'total_sum': 0, 'total_count': 0}
        global_avg_data[metric]['total_sum'] += sum_val
        global_avg_data[metric]['total_count'] += count_val

print("Global Average per Metric (from Shards):")
for metric, data in global_avg_data.items():
    avg = data['total_sum'] / data['total_count']
    print(f"  {metric}: {avg:.3f}")

# Compare to single-DB answer
print("\nSingle-DB Average per Metric:")
single_db_results = con.execute('SELECT metric, AVG(value) FROM telemetry GROUP BY metric').fetchall()
for metric, avg_val in single_db_results:
    print(f"  {metric}: {avg_val:.3f}")


# 2. Show the downside of a bad shard key: sharding by metric
print("\n--- Sharding by Metric (Skew Demonstration) ---")

def shard_of_metric(metric):
    # Using a simple hash for demonstration, for real world, map metrics to shards explicitly
    return zlib.crc32(metric.encode()) % N_SHARDS

metric_shards = [sqlite3.connect(f'metric_shard_{i}.db') for i in range(N_SHARDS)]
for msc in metric_shards:
    msc.execute('DROP TABLE IF EXISTS telemetry')
    msc.execute('CREATE TABLE telemetry (machine_id TEXT, metric TEXT, value REAL)')

# Re-read all rows from the original connection
all_rows = con.execute('SELECT machine_id, metric, value FROM telemetry').fetchall()

for mid, metric, val in all_rows:
    metric_shards[shard_of_metric(metric)].execute('INSERT INTO telemetry VALUES (?,?,?)', (mid, metric, val))
for msc in metric_shards: msc.commit()

metric_shard_sizes = [msc.execute('SELECT COUNT(*) FROM telemetry').fetchone()[0] for msc in metric_shards]
print('Rows per metric shard:', metric_shard_sizes)
print('This demonstrates significant skew, as metrics are unevenly distributed, leading to hotspots.')


--- Global Average (Scatter-Gather) ---
Global Average per Metric (from Shards):
  current: 30.974
  pressure: 6.195
  temp: 68.006
  vibration: 2.598

Single-DB Average per Metric:
  current: 30.974
  pressure: 6.195
  temp: 68.006
  vibration: 2.598

--- Sharding by Metric (Skew Demonstration) ---
Rows per metric shard: [0, 59810, 60033, 80158]
This demonstrates significant skew, as metrics are unevenly distributed, leading to hotspots.



# 3. shard-key rule:
- The rule for picking a good shard key is to choose a column (or combination of columns) that:
 - Has high cardinality: There should be a large number of unique values to ensure data can be spread across many shards.
 - Ensures even distribution: The values should be distributed as evenly as possible across the shards to prevent hotspots (where one shard handles significantly more data or traffic than others).
 - Is frequently used in queries: Sharding by a column commonly used in WHERE clauses allows queries to be routed to a single shard, avoiding expensive scatter-gather operations.

#5. Materialized view & security

In [11]:
# -----------------------------------------------------------
# 🔹 5A. MATERIALIZED VIEW: precompute an expensive rollup
# -----------------------------------------------------------
rollup_sql = '''SELECT machine_id, metric, AVG(value) avg_v, COUNT(*) n
                FROM telemetry GROUP BY machine_id, metric'''
t_live = timed(rollup_sql)                       # computed on the fly every time
con.execute('DROP TABLE IF EXISTS mv_machine_metric')
con.execute('CREATE TABLE mv_machine_metric AS ' + rollup_sql); con.commit()
con.execute('CREATE INDEX idx_mv ON mv_machine_metric(machine_id)'); con.commit()
t_mv = timed('SELECT * FROM mv_machine_metric WHERE machine_id=?', ('M007',))
print(f'aggregation live: {t_live:.2f} ms  |  read from materialized view: {t_mv:.2f} ms')
print('Trade-off: the view must be REFRESHED when underlying data changes (it can go stale).')

aggregation live: 70.55 ms  |  read from materialized view: 0.02 ms
Trade-off: the view must be REFRESHED when underlying data changes (it can go stale).


In [12]:
# -----------------------------------------------------------
# 🔹 5B. SECURITY: mask PII-like fields & a read-only connection
# -----------------------------------------------------------
# tokenize an identifier so analysts can join without seeing the raw id
def tokenize(s, salt='plant2024'):
    return hashlib.sha256((salt + str(s)).encode()).hexdigest()[:12]
print('M007 tokenized ->', tokenize('M007'))
# a read-only connection: writes are rejected (least privilege)
ro = sqlite3.connect('file:plant.db?mode=ro', uri=True)
try:
    ro.execute('DELETE FROM telemetry'); print('write succeeded (unexpected!)')
except Exception as e:
    print('write blocked on read-only connection:', type(e).__name__)

M007 tokenized -> 936b3bb31c50
write blocked on read-only connection: OperationalError


#### 🧪 EXERCISE 5 — Keyset pagination
Offset pagination (`LIMIT n OFFSET k`) gets slower as `k` grows because the DB still scans the skipped rows.
1. Time `... ORDER BY reading_id LIMIT 50 OFFSET 150000` vs **keyset** pagination `... WHERE reading_id > 150000 ORDER BY reading_id LIMIT 50`.
2. In a comment, explain why keyset pagination stays fast on deep pages (it seeks via the index instead of counting past skipped rows).

In [13]:
# 1. offset vs keyset pagination timing
# Offset pagination
print("--- Pagination Timing ---")
t_offset = timed('SELECT * FROM telemetry ORDER BY reading_id LIMIT 50 OFFSET 150000')
print(f'Offset pagination (LIMIT 50 OFFSET 150000): {t_offset:.2f} ms')

# Keyset pagination
t_keyset = timed('SELECT * FROM telemetry WHERE reading_id > 150000 ORDER BY reading_id LIMIT 50')
print(f'Keyset pagination (WHERE reading_id > 150000 ORDER BY reading_id LIMIT 50): {t_keyset:.2f} ms')

print(f'Keyset pagination is {t_offset/max(t_keyset, 1e-6):.1f}x faster than offset pagination.')


--- Pagination Timing ---
Offset pagination (LIMIT 50 OFFSET 150000): 233.37 ms
Keyset pagination (WHERE reading_id > 150000 ORDER BY reading_id LIMIT 50): 29.54 ms
Keyset pagination is 7.9x faster than offset pagination.


# 2. why keyset wins on deep pages:
- Keyset pagination (also known as cursor-based pagination) is faster on deep pages because it leverages an index directly.
- Instead of scanning and discarding a large number of rows (as OFFSET does), it directly seeks to the next starting point using the last seen key from the previous page.
- This means the database doesn't have to count or sort through all the skipped rows, making it much more efficient, especially for very large offsets.

#📘 Summary

| Technique | What you measured |
| --------- | ----------------- |
| Indexing | scan → index search, large speedup |
| Cache-aside | repeated reads served from memory |
| Read replicas | offload reads; accept replication lag |
| Sharding | spread data by key; route or scatter-gather |
| Materialized view | precompute rollups; refresh to stay fresh |
| Security | tokenize PII; read-only least-privilege access |

**Core lesson:** production performance comes from a toolbox of trade-offs — cache hot reads, index what you filter, replicate and shard to scale out, precompute expensive rollups — each buying speed or scale at the cost of freshness, complexity or write overhead.